In [1]:
import pandas as pd
from xgboost import XGBClassifier as xg
from sklearn.model_selection import train_test_split as tts
from sklearn.metrics import accuracy_score as AS
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
passenger_ids = test['PassengerId']
def process(df):
    num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    for col in num_cols: 
       df[col] = df[col].fillna(df[col].median())
    for col in cat_cols:
       df[col] = df[col].fillna(df[col].mode()[0])
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)
    df['CryoSleep'] = df['CryoSleep'].map({True:1, False:0})
    df['VIP'] = df['VIP'].map({True:1, False:0})
    df['Cabin'] = df['Cabin'].fillna('U/0/U')
    df['Deck'] = df['Cabin'].str.split('/').str[0]
    df['Side'] = df['Cabin'].str.split('/').str[2]
    df = pd.get_dummies(df, columns=['HomePlanet', 'Destination', 'Deck', 'Side'])
    df = df.drop(columns=['PassengerId', 'Cabin', 'Name'])
    return df
train = process(train)
test = process(test)
train['Transported'] = train['Transported'].map({True:1, False:0})
X = train.drop(columns=['Transported'])
y = train['Transported']
X_train, X_val, y_train, y_val = tts(X, y, test_size=0.2, random_state=3)
model = xg(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=3)
model.fit(X_train, y_train)
val_pred = model.predict(X_val)
print("准确率", AS(y_val, val_pred))
test_pred = model.predict(test)
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Transported': test_pred.astype(bool)
})
submission.to_csv('submission.csv', index=False)

/tmp/ipykernel_17/3759524603.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])
/tmp/ipykernel_17/3759524603.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


准确率 0.8027602070155262
